In [1]:
from transformers import AutoTokenizer, BertForMaskedLM
from datasets import load_dataset, Dataset, DatasetDict
import torch
import pandas as pd
import numpy as np
from typing import Any, Union, List, Tuple, Dict
from transformers.data.data_collator import DataCollatorMixin, InputDataClass
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer

In [2]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
# df['answers.text'] = df['answers.text'].apply(lambda x: [x])

In [3]:
df_train = df[df['mode']=='train']
df_val = df[df['mode']=='val']
df_test = df[df['mode']=='test']

In [4]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = BertForMaskedLM.from_pretrained("google-bert/bert-base-uncased")

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        item = {
                'input_ids': self.df.iloc[idx]['answers.text'], 
                'labels': eval(self.df.iloc[idx].assetlongdescription_entity_llms)
               }
        return item

In [6]:
ds_train = CustomDataset(df_train)
ds_val = CustomDataset(df_val)
ds_test = CustomDataset(df_test)

In [7]:
def collate_fn(data, mask_prob=0.3):
    batch = {'original_text': [], 'masked_text': []}
    for i in range(len(data)):
        text = data[i]['input_ids']    
        batch['original_text'].append(text)
        entities = np.array(data[i]['labels'])
        original_text = text
        sel_idxs = np.random.binomial(1, mask_prob, len(entities)).astype(bool)
        masked_entities = entities[sel_idxs]
    original_tokenized = tokenizer(batch['original_text'], padding="max_length", 
                                   max_length=512, truncation=True, return_tensors='pt')
    mask_tokenized = original_tokenized['input_ids'].detach().clone()
    mask_token_id = torch.tensor(tokenizer.mask_token_id, dtype=torch.long)
    all_labels = []
    for idx, original_tokens in enumerate(original_tokenized['input_ids']):
        entities = np.array(data[idx]['labels'])
        original_text = text
        sel_idxs = np.random.binomial(1, mask_prob, len(entities)).astype(bool)
        masked_entities = entities[sel_idxs]
        mask_idxs = torch.zeros(len(original_tokens)).bool()
        for entity in masked_entities:
            tokenized_entity = tokenizer(entity, return_tensors='pt')['input_ids'][0, 1:-1]
            for i in range(len(original_tokens) - len(tokenized_entity) + 1):
                if torch.equal(original_tokens[i], mask_token_id):
                    break
                original_token_slice = original_tokens[i:i+len(tokenized_entity)]
                if torch.equal(original_token_slice, tokenized_entity):
                    mask_idxs[i:i+len(tokenized_entity)] = True
        labels = torch.where(mask_idxs, original_tokens, -100)
        mask_tokenized[idx][mask_idxs] = mask_token_id
        all_labels.append(labels)
    results = {
        'input_ids': mask_tokenized,
        'labels': torch.stack(all_labels),
        'token_type_ids': original_tokenized['token_type_ids']
    }
    return results

In [8]:
dataloader = DataLoader(ds_train, batch_size=4, collate_fn=collate_fn)

In [9]:
batch = next(iter(dataloader))

In [10]:
print("*"*20+" Input "+"*"*20)
print(tokenizer.decode(batch['input_ids'][0]).replace('[PAD]', ''))
print("*"*20+" Label "+"*"*20)
labels = batch['labels'][0][batch['labels'][0] != -100]
print(tokenizer.batch_decode(labels))

******************** Input ********************
[CLS] the equipment accumulator - pneumatic - [MASK] type, is categorized as fixed asset and has the following boundary : a pneumatic accumulator - [MASK] type in this database is comprised of : - tank - [MASK] - air line check valve, if present - gas precharge valve [SEP]                                                                                                                                                                                                                                                                                                                                                                                                                                                                   
******************** Label ********************
['bladder', 'bladder', 'bladder']


In [11]:
outputs = model(**batch)

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


In [12]:
outputs.loss.item()

7.952976226806641

In [13]:
training_args = TrainingArguments(
    output_dir="entity_masking_mlm",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collate_fn,
)

trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


Epoch,Training Loss,Validation Loss



KeyboardInterrupt



In [ ]:
trainer.train()